#Instructions

In [ ]:
'''

This script provides a generic example of how the user can load our pretrained models to predict
metabolite levels directly from RNA sequencing data.

- The RNA sequencing data must be in TPM format (csv file, sample names in rows, gene names in columns).
- The name of the predicted metabolites must be loaded by the provided file,
as the saved model does not include output names.

'''

#Import modules

In [ ]:
!pip install joblib
!pip install pandas
!pip install numpy
!pip install pickle

In [ ]:
import pandas as pd
import numpy as np
import joblib
import pickle

#Metabolite prediction

In [ ]:
#Load a pre-trained model

model = joblib.load('ElasticNet.pkl')

#Get model features
model_features = model.feature_names_in_
print(model_features)

#Load y features
with open('y_train_features.pkl', 'rb') as f:
    y_train_features_loaded = pickle.load(f)

print(y_train_features_loaded)

In [ ]:
#Load RNA-Seq data (already TPM normalized)

rna = pd.read_csv('rna_tpm.csv')
rna

In [ ]:
#Apply log2 transformation (Optional)

rna = rna.apply(lambda x: np.log2(x + 1))
rna

In [ ]:
#Model features not found in data are added as NaN values
features_not_in_data = list(set(model_features) - set(rna.columns))
for feature in features_not_in_data:
  rna[feature] = np.nan

#Data filtered to include model features
filtered_data = rna[model_features]
filtered_data

In [ ]:
#Predict metabolite levels

predictions = model.predict(filtered_data)
predictions = pd.DataFrame(predictions, index = filtered_data.index, columns = y_train_features_loaded)
predictions

In [ ]:
#Save predictions in a csv file

predictions.to_csv('predictions.csv', index = True)